# Test des règles de rejet

On ajoute **13 lignes fausses** dans une **copie** des CSV, on relance le pipeline et on vérifie qu'elles sont rejetées. Les données originales et le Data Warehouse du projet ne sont pas modifiés.

Version notebook de `test_rejets.py`.

In [1]:
from pathlib import Path

# Un notebook n'a pas de __file__ : on remonte les dossiers jusqu'à la racine du projet
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "03_data" / "raw").is_dir())
print("Projet :", ROOT.name)

Projet : SmartPort_Projet01


## 1. Préparer un dossier temporaire et charger le pipeline

In [2]:
# Bibliothèques, dossier temporaire et chargement des fonctions du pipeline
import os, sys, shutil, sqlite3, io, contextlib, tempfile
import pandas as pd
HERE = str(ROOT / "04_scripts" / "etl")
DATA_DIR = str(ROOT / "03_data" / "raw")
DOSSIER_INITIAL = os.getcwd()
WORK = tempfile.mkdtemp(prefix="smartport_test_")
sys.path.insert(0, HERE)
# etl_pipeline s'execute a l'import (pipeline normal, base temporaire) : on masque sa sortie
with contextlib.redirect_stdout(io.StringIO()):
    anciens_argv = sys.argv
    sys.argv = ["etl_pipeline.py", DATA_DIR, os.path.join(WORK, "normal.db")]
    import etl_pipeline as E
    sys.argv = anciens_argv
rejouer_pipeline, SOURCES = E.rejouer_pipeline, E.SOURCES
os.chdir(WORK)

## 2. Copier les CSV

In [3]:
# Copie des CSV dans le dossier "bad" : les originaux ne sont pas touchés
os.makedirs("bad", exist_ok=True)
for s in SOURCES:
    shutil.copy(f"{DATA_DIR}/{s}.csv", f"bad/{s}.csv")

e = pd.read_csv("bad/container_events.csv")
m = pd.read_csv("bad/yard_moves.csv")
b = e.iloc[0].to_dict()
def ev(**kw): d = dict(b); d.update(kw); return d

## 3. Ajouter les lignes fausses (événements puis mouvements)

In [4]:
# Chaque ligne fausse est une copie d'une vraie ligne, avec une erreur injectée
# (le code de la règle attendue est écrit à droite)
mauvais_events = [
    ev(event_id=e.event_id.iloc[1]),                 # EV02 : identifiant en double
    ev(event_id=99001, event_ts="pas-une-date"),     # EV03 : date illisible
    ev(event_id=99002, event_type="TELEPORT"),       # EV04 : type inconnu
    ev(event_id=99003, freight_status="XXX"),        # EV05 : statut invalide
    ev(event_id=99004, gross_weight_kg=-50),         # EV06 : poids négatif
    ev(event_id=99005, gross_weight_kg=99999),       # EV06 : poids trop grand
    ev(event_id=99006, customer_id="C_INCONNU"),     # EV07 : client inexistant
    ev(event_id=99007, vessel_id="V_INCONNU"),       # EV08 : navire inexistant
    ev(event_id=99008, yard_area=""),                # EV09 : zone vide
]
mb = m.iloc[0].to_dict()
mauvais_moves = [
    dict(mb, move_id=m.move_id.iloc[1]),                    # MV02 : identifiant en double
    dict(mb, move_id=88001, move_ts="31/31/2026"),          # MV03 : date illisible
    dict(mb, move_id=88002, equipment_id="EQ_INCONNU"),     # MV04 : équipement inexistant
    dict(mb, move_id=88003, container_id="CONT_INCONNU"),   # MV06 : conteneur inexistant
]
pd.concat([e, pd.DataFrame(mauvais_events)]).to_csv("bad/container_events.csv", index=False)
pd.concat([m, pd.DataFrame(mauvais_moves)]).to_csv("bad/yard_moves.csv", index=False)

## 4. Relancer le pipeline sur les données fausses

In [5]:
# Pipeline complet sur les CSV faux, dans une base de test séparée
if os.path.exists("smartport_test.db"): os.remove("smartport_test.db")
statuts = rejouer_pipeline("bad", "smartport_test.db")
print("Contrôles en échec (FAIL) sur les données défectueuses :", statuts.count("FAIL"), "(attendu : ils détectent les erreurs injectées)")

Contrôles en échec (FAIL) sur les données défectueuses : 4 (attendu : ils détectent les erreurs injectées)


## 5. Résultats

Les contrôles en échec sont **voulus** : ils montrent que les erreurs sont détectées.

In [6]:
# Table etl_rejects : une ligne par rejet, avec le code de la règle
tc = sqlite3.connect("smartport_test.db")
print("Lignes rejetées, par règle :")
display(pd.read_sql("SELECT rule_id, rule_description, COUNT(*) AS nb FROM etl_rejects GROUP BY 1, 2 ORDER BY 1", tc))

Lignes rejetées, par règle :


,rule_id,rule_description,nb
0,EV02,event_id en double (on garde la 1re ligne),1
1,EV03,event_ts illisible,1
2,EV04,event_type hors domaine,1
3,EV05,freight_status hors FCL/MTY,1
4,EV06,poids non numerique ou hors 0..40000 kg,2
5,EV07,customer_id absent du referentiel,1
6,EV08,vessel_id absent du referentiel,1
7,EV09,yard_area manquante,1
8,MV02,move_id en double (on garde la 1re ligne),1
9,MV03,move_ts illisible,1


**Interprétation.** Les 13 lignes fausses sont toutes rejetées (la somme de la colonne `nb` fait 13 ; EV06 compte 2 lignes : un poids négatif et un poids trop grand). Chaque rejet garde le code de la règle, donc on sait **pourquoi** la ligne a été écartée. Ce test couvre 12 des 15 règles : EV01, MV01 et MV05 ne sont pas testées.

In [7]:
# Vérifier que les lignes fausses ne sont pas entrées dans les tables de faits
print("Faits chargés malgré les 13 erreurs injectées :")
display(pd.read_sql("""SELECT (SELECT COUNT(*) FROM fact_container_event) AS evenements,
                       (SELECT COUNT(*) FROM fact_yard_move) AS mouvements""", tc))

Faits chargés malgré les 13 erreurs injectées :


,evenements,mouvements
0,9000,4500


**Interprétation.** On retrouve 9 000 événements et 4 500 mouvements, comme dans les données originales : les lignes fausses ne sont pas entrées dans les tables de faits.

In [8]:
# Afficher les contrôles dont le statut n'est pas PASS
print("Contrôles qui détectent les erreurs injectées (donnée source) :")
display(pd.read_sql("SELECT check_id, layer, status, observed FROM etl_quality_checks WHERE status <> 'PASS' ORDER BY layer, check_id", tc))

Contrôles qui détectent les erreurs injectées (donnée source) :


,check_id,layer,status,observed
0,Q13,CORE,WARN,712.0
1,DQ_DUPLICATE_CHECK_container_events,RAW,FAIL,1.0
2,DQ_DUPLICATE_CHECK_yard_moves,RAW,FAIL,1.0
3,FK_CONTAINER,RAW,FAIL,1.0
4,FK_EQUIPMENT,RAW,FAIL,1.0
5,Q15,STAGING,INFO,1481.0


**Interprétation.** Les 4 `FAIL` sont voulus. Ils signalent dans la couche RAW les erreurs ajoutées : un événement en double, un mouvement en double, un conteneur inconnu et un équipement inconnu. `Q13` (712 mouvements avec la même zone de départ et d'arrivée) et `Q15` (1 481 zones rattachées à la zone parente) ne viennent pas des lignes ajoutées : ils existent déjà dans les données normales.

In [9]:
# Q09 et Q10 : RAW - (chargées + rejetées) doit être égal à 0
print("Réconciliation (écart attendu 0) :")
display(pd.read_sql("SELECT check_id, status, observed FROM etl_quality_checks WHERE check_id IN ('Q09','Q10')", tc))

Réconciliation (écart attendu 0) :


,check_id,status,observed
0,Q09,PASS,0.0
1,Q10,PASS,0.0


**Interprétation.** `Q09` et `Q10` sont à 0 : pour les événements et les mouvements, lignes RAW = lignes chargées + lignes rejetées. Aucune ligne ne disparaît sans trace.

In [10]:
# Fermer la base de test et revenir au dossier de départ
tc.close()
os.chdir(DOSSIER_INITIAL)

## Conclusion

Les règles testées écartent les lignes fausses, notent la raison du rejet et gardent un écart de réconciliation nul. Les données originales ne sont pas modifiées. Il reste à tester les règles EV01, MV01 et MV05.